In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Install all packages from
!pip install -r ../requirements.txt

In [ ]:
import json, os, csv
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
df = pd.read_csv("../stats/download_stats.csv")
df.head()

In [ ]:
# Count unique recids in lists/datasets
folder = "../lists"
res = set()
for filename in os.listdir(folder):
    if filename.endswith(".json") and filename.startswith('delphi-datasets'):
        with open(os.path.join(folder, filename), 'r', encoding='utf-8') as f:
            try:
                data = json.load(f)
                recids = set()
                def extract_recids(obj):
                    if isinstance(obj, dict):
                        for key, value in obj.items():
                            if key in ('recid', 'id'):
                                try:
                                    recids.add(int(value))
                                except (TypeError, ValueError):
                                    pass
                            extract_recids(value)
                    elif isinstance(obj, list):
                        for item in obj:
                            extract_recids(item)
                extract_recids(data)
                res.update(recids)
            except json.JSONDecodeError:
                pass
print(f"Unique recids in lists/datasets: {len(res)}")

In [ ]:
# find recid in res
recid = 93080
if recid in res:
    print(f"Recid {recid} found in lists/datasets.")
else:
    print(f"Recid {recid} NOT found in lists/datasets.")

In [ ]:
with open('delphi_records_master.json', 'r') as cache_file:
    master_cache = json.load(cache_file)

In [ ]:
# find recid x in master_cache
recid = 85240
recid_str = str(recid)
if recid_str in master_cache:
    print(f"Recid {recid} found in master_cache.")
else:
    print(f"Recid {recid} NOT found in master_cache.")

# find recid x in lists
if recid in res:
    print(f"Recid {recid} found in lists/datasets.")
else:
    print(f"Recid {recid} NOT found in lists/datasets.")

# find recid x df
if recid in df['recid'].values:
    print(f"Recid {recid} found in {len(df[df['recid'] == recid])} download records.")
else:
    print(f"Recid {recid} NOT found in download stats.")

In [ ]:
# Print metadata of recid x from master_cache
metadata = master_cache.get(recid_str)
if metadata:
    print(f"Metadata for recid {recid} in master_cache:")
    print(json.dumps(metadata, indent=2))

In [ ]:
# print all recids with errors
error_recids = df[df['success'] == False]['recid'].unique()
print(len(error_recids))
print("Recids with errors:")
# for recid in error_recids:
#     print(recid)

In [ ]:
# group by file print number of errors and successes per file
file_stats = df.groupby('file')['success'].value_counts().unstack(fill_value=0)
# filter False more than 0
file_stats = file_stats[file_stats[False] > 0]
file_stats = file_stats[file_stats[True] == 0]
print(len(file_stats))
print(file_stats)

In [ ]:
# what are the errors for recid 84179
recid_84179_errors = df[df['recid'] == 84179]['error'].value_counts()
print(recid_84179_errors)